# SVM: Optimasi dan Submission

Model SVM RBF from scratch ini menggunakan optimasi dual working-set dan cache kernel. Pembanding SVC digunakan hanya untuk evaluasi. Eksplorasi awal menggunakan data training berlabel; test Kaggle tidak digunakan untuk memilih model.

## Persiapan

Jalankan semua sel secara berurutan dari kernel baru. Paket yang diperlukan: NumPy, pandas, scikit-learn dan Jupyter. Proses dapat memakan beberapa menit dan menggunakan cache kernel hingga 512 MiB.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p / "data/train.csv").is_file() and (p / "src/svm_optimized.py").is_file()), None)
if root is None:
    root = next((p / "IF3170_G31_KelompokDadakan" for p in candidates if (p / "IF3170_G31_KelompokDadakan/data/train.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("Buka notebook dari dalam folder proyek atau folder induknya")
sys.path.insert(0, str(root / "src"))
import svm_optimized as svm
svm.configure_baseline()
from svm_optimized import load_data, validate, make_submission, CONFIG

X, y, X_test, test, sample = load_data()
print("Train:", X.shape, "Test:", X_test.shape)
print("Konfigurasi:", CONFIG)

## Validasi SVM

Pada masing-masing seed, 20% data disisihkan untuk validasi. Dari sisanya, 20% digunakan memilih threshold berdasarkan Macro F1. Preprocessor hanya di-fit pada bagian training model. Dua hasil validasi dapat saling beririsan dan pemilihan hyperparameter didasarkan pada eksperimen sebelumnya, sehingga hasil ini bukan estimasi final yang benar-benar independen dari pemilihan model.

In [ ]:
hasil = validate(X, y)
display(hasil.round(4))
from IPython.display import Image, display
display(Image(filename=str(root / "output" / "SVM" / "optimized" / "validation_macro_f1.png")))

## Latih pada seluruh data berlabel dan buat submission

Threshold final adalah median dari threshold kalibrasi kedua split. Preprocessor dan model scratch di-fit pada seluruh `train.csv`; `test.csv` hanya ditransformasi dan diprediksi. Model dan preprocessor disimpan, dimuat ulang, dan prediksinya dibandingkan sebelum CSV ditulis.

Kandidat kedua `submission_svm_threshold0.csv` memakai threshold standar 0 dan dapat dibandingkan dengan kandidat threshold kalibrasi. Pilihan threshold ini ikut dilihat pada dua split validasi, sehingga nilainya bukan evaluasi yang sepenuhnya independen.

In [ ]:
submission, metadata = make_submission(X, y, X_test, test, sample, hasil)
display(submission.head())
print(metadata)

# Kandidat tambahan dengan threshold standar 0.
from svm_optimized import make_zero_threshold_submission
submission_zero = make_zero_threshold_submission(X_test, test, sample)
display(submission_zero.head())

Output ada di `output/SVM/optimized/`: CSV submission, tabel validasi, model scratch, preprocessor, dan metadata. Skor Kaggle hanya diketahui setelah CSV diunggah. Jangan memuat file pickle dari sumber yang tidak dipercaya.